# COVID-19 Analysis Example Notebook

This notebook demonstrates basic usage of the COVID-19 analysis tools.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Data

In [ ]:
# Load COVID-19 data
df = pd.read_csv('owid-covid-data.csv', parse_dates=['date'])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Countries: {df['location'].nunique()}")

## Example 1: Explore a Single Country

In [ ]:
# Select a country
country = 'United States'
country_data = df[df['location'] == country].copy()

# Plot cases and deaths
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.plot(country_data['date'], country_data['new_cases_smoothed_per_million'], label='Cases per Million')
ax1.set_title(f'{country} - New Cases per Million (7-day average)')
ax1.set_ylabel('Cases per Million')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(country_data['date'], country_data['new_deaths_smoothed_per_million'], label='Deaths per Million', color='red')
ax2.set_title(f'{country} - New Deaths per Million (7-day average)')
ax2.set_xlabel('Date')
ax2.set_ylabel('Deaths per Million')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Example 2: Healthcare Strain Analysis

In [ ]:
# Plot ICU utilization vs deaths
country_data_clean = country_data[['icu_patients_per_million', 'new_deaths_smoothed_per_million']].dropna()

if len(country_data_clean) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.scatter(country_data_clean['new_deaths_smoothed_per_million'], 
               country_data_clean['icu_patients_per_million'],
               alpha=0.5)
    
    ax.set_xlabel('New Deaths per Million (7-day avg)')
    ax.set_ylabel('ICU Patients per Million')
    ax.set_title(f'{country} - ICU Utilization vs Deaths')
    ax.grid(True, alpha=0.3)
    
    # Calculate correlation
    corr = country_data_clean['new_deaths_smoothed_per_million'].corr(
        country_data_clean['icu_patients_per_million']
    )
    ax.text(0.05, 0.95, f'Correlation: {corr:.3f}', 
            transform=ax.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.show()
else:
    print(f"No ICU data available for {country}")

## Example 3: Pandemic Fatigue Detection

In [ ]:
# Define fatigue indicator
country_data['case_14d_avg'] = country_data['new_cases_smoothed_per_million'].rolling(14, min_periods=7).mean()
country_data['case_change'] = country_data['case_14d_avg'].pct_change(periods=14)
country_data['high_stringency'] = country_data['stringency_index'] >= 60
country_data['rising_cases'] = country_data['case_change'] > 0.2
country_data['fatigue'] = country_data['high_stringency'] & country_data['rising_cases']

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Stringency and cases
ax1_twin = ax1.twinx()
ax1.plot(country_data['date'], country_data['stringency_index'], label='Stringency Index', color='blue')
ax1_twin.plot(country_data['date'], country_data['new_cases_smoothed_per_million'], 
              label='Cases per Million', color='red', alpha=0.7)

# Mark fatigue periods
fatigue_dates = country_data[country_data['fatigue']]['date']
for date in fatigue_dates:
    ax1.axvline(date, color='orange', alpha=0.3, linewidth=0.5)

ax1.set_ylabel('Stringency Index', color='blue')
ax1_twin.set_ylabel('Cases per Million', color='red')
ax1.set_title(f'{country} - Pandemic Fatigue Detection')
ax1.legend(loc='upper left')
ax1_twin.legend(loc='upper right')

# Fatigue indicator
ax2.fill_between(country_data['date'], 0, country_data['fatigue'].astype(int), 
                 alpha=0.5, color='orange', label='Fatigue Periods')
ax2.set_xlabel('Date')
ax2.set_ylabel('Fatigue Indicator')
ax2.set_ylim(-0.1, 1.1)
ax2.legend()

plt.tight_layout()
plt.show()

# Summary
fatigue_days = country_data['fatigue'].sum()
total_days = len(country_data)
fatigue_pct = (fatigue_days / total_days) * 100

print(f"\nFatigue Summary for {country}:")
print(f"  Fatigue days: {fatigue_days}")
print(f"  Total days: {total_days}")
print(f"  Fatigue percentage: {fatigue_pct:.1f}%")

## Example 4: Compare Multiple Countries

In [ ]:
# Select countries to compare
countries = ['United States', 'United Kingdom', 'Germany', 'France']

fig, ax = plt.subplots(figsize=(12, 6))

for country in countries:
    country_data = df[df['location'] == country]
    ax.plot(country_data['date'], country_data['new_cases_smoothed_per_million'], 
            label=country, linewidth=2)

ax.set_xlabel('Date')
ax.set_ylabel('New Cases per Million (7-day avg)')
ax.set_title('COVID-19 Cases Comparison')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Example 5: Summary Statistics

In [ ]:
# Calculate summary statistics for selected countries
summary_data = []

for country in countries:
    country_df = df[df['location'] == country]
    
    summary_data.append({
        'Country': country,
        'Total Cases': country_df['total_cases'].max(),
        'Total Deaths': country_df['total_deaths'].max(),
        'Peak Cases/M': country_df['new_cases_smoothed_per_million'].max(),
        'Peak Deaths/M': country_df['new_deaths_smoothed_per_million'].max(),
        'Avg Stringency': country_df['stringency_index'].mean()
    })

summary_df = pd.DataFrame(summary_data)
summary_df

## Next Steps

- Explore the interactive dashboard: `streamlit run dashboard/app.py`
- Run full analysis scripts in `scripts/` folder
- Read detailed reports in `docs/` folder
- Customize analyses for your research questions